# Loan Approval Prediction — Logistic Regression

Predicts whether a loan application will be **approved (Y)** or **rejected (N)** based on applicant
demographics, income, and credit history.

**Dataset:** `data/loan_data.csv` — 614 rows, 13 columns, matching the schema of the well-known
Analytics Vidhya / Kaggle *Loan Prediction Practice Problem* dataset. This copy is synthetically
generated (see `data/generate_data.py`) since the original sits behind a Kaggle login wall; column
names and structure are identical, so a real download can be swapped in without changing any code.

**Model:** Logistic Regression (`scikit-learn`)


## 1. Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    classification_report, confusion_matrix, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
RANDOM_STATE = 42
%matplotlib inline

## 2. Load the data

In [ ]:
df = pd.read_csv("data/loan_data.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False)

## 3. Exploratory Data Analysis

In [ ]:
df["Loan_Status"].value_counts(normalize=True).plot(
    kind="bar", color=["#2a9d8f", "#e76f51"], title="Loan Status Distribution")
plt.ylabel("Proportion")
plt.xticks([0, 1], ["Approved (Y)", "Rejected (N)"], rotation=0)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Credit_History", "Education", "Property_Area"]):
    pd.crosstab(df[col], df["Loan_Status"], normalize="index").plot(
        kind="bar", stacked=True, ax=ax, color=["#e76f51", "#2a9d8f"])
    ax.set_title(f"Approval Rate by {col}")
    ax.legend(["Rejected", "Approved"], fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["ApplicantIncome"], bins=30, ax=axes[0], color="#264653")
axes[0].set_title("Applicant Income Distribution")
sns.histplot(df["LoanAmount"].dropna(), bins=30, ax=axes[1], color="#264653")
axes[1].set_title("Loan Amount Distribution")
plt.tight_layout()
plt.show()

**Observations:**
- Roughly 3 in 4 applications are approved in this dataset.
- `Credit_History` shows the strongest visible relationship with approval — applicants with a
  positive credit history are approved far more often.
- Income and loan amount are both right-skewed, which is why we apply a log transform later.

## 4. Data Cleaning & Preprocessing

In [ ]:
df = df.drop(columns=["Loan_ID"])

for col in ["Gender", "Married", "Dependents", "Self_Employed", "Credit_History", "Loan_Amount_Term"]:
    df[col] = df[col].fillna(df[col].mode()[0])
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

df["Dependents"] = df["Dependents"].replace("3+", "3").astype(int)

### Feature engineering

In [ ]:
df["TotalIncome"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["LoanAmount_Log"] = np.log1p(df["LoanAmount"])
df["TotalIncome_Log"] = np.log1p(df["TotalIncome"])

df[["TotalIncome", "LoanAmount_Log", "TotalIncome_Log"]].head()

### Encode categorical variables

In [ ]:
cat_cols = ["Gender", "Married", "Education", "Self_Employed", "Property_Area"]
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df_encoded["Loan_Status"] = df_encoded["Loan_Status"].map({"Y": 1, "N": 0})
df_encoded.head()

### Correlation heatmap

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_encoded.corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Feature Correlation Heatmap")
plt.show()

## 5. Train/Test Split & Scaling

In [ ]:
X = df_encoded.drop(columns=["Loan_Status"])
y = df_encoded["Loan_Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 6. Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

## 7. Evaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC:  {auc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Rejected", "Approved"]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred), display_labels=["Rejected", "Approved"]
).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax)
ax.set_title("ROC Curve")
plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
coefs = pd.Series(model.coef_[0], index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 6))
coefs.plot(kind="barh", ax=ax, color=np.where(coefs > 0, "#2a9d8f", "#e76f51"))
ax.set_title("Feature Impact on Approval (Logistic Regression Coefficients)")
ax.set_xlabel("Coefficient (standardized)")
plt.tight_layout()
plt.show()

**Interpretation:** `Credit_History` dominates the model's decision, consistent with the EDA.
Positive coefficients push toward approval; negative coefficients push toward rejection.

## 9. Export Predictions

In [ ]:
results = X_test.copy()
results["Actual"] = y_test.map({1: "Y", 0: "N"}).values
results["Predicted"] = pd.Series(y_pred, index=X_test.index).map({1: "Y", 0: "N"})
results["Approval_Probability"] = np.round(y_proba, 3)

results.to_csv("outputs/predictions.csv", index=False)
results.head(10)

## 10. Conclusion

The logistic regression model achieves solid accuracy and AUC using a simple, fully interpretable
linear model. `Credit_History` is by far the strongest predictor of loan approval, followed by
total household income and property area. Because logistic regression outputs coefficients directly
tied to standardized features, this model is well suited to settings where explainability matters
(e.g. regulated lending decisions).